# Sampling light-curve parameters with `BinaryCircularParameterization`

This notebook demonstrates how to use a `Parameterization` class to sample an
MCMC chain in the light-curve parameter space while evaluating the Galactic
prior in physical space.

The binary circular parameterization maps 12 light-curve parameters

```
theta = [t0, tE, u0, rho, q, s, alpha, piEN, piEE, gamma1, gamma2, gamma3]
```

to the five physical parameters `(ML, DL, DS, mu_N, mu_E)` expected by
`GalacticPrior`, and supplies the log-Jacobian of that mapping so the
probability density is correctly transferred between the two coordinate
systems.

**Prerequisites**: run `pre_runner.ipynb` first to generate the histogram files
in `example/pre_runner_outputs/demo/`.

**Units**
- `t0`, `tE`: days (HJD)
- `rho`: source-radius ratio (dimensionless)
- `q`: mass ratio (dimensionless)
- `s`: projected separation in Einstein radii
- `alpha`: trajectory angle in radians
- `piEN`, `piEE`: microlensing parallax components
- `gamma1`, `gamma2`, `gamma3`: orbital motion parameters
- `DL`, `DS`: kpc  |  `mu_N`, `mu_E`: mas/yr

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src" / "gapmoe").exists():
    repo_root = cwd
elif (cwd.parent / "src" / "gapmoe").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repository root or example directory.")

sys.path.insert(0, str(repo_root / "src"))

import emcee
import matplotlib.pyplot as plt
import numpy as np

from gapmoe import GalacticPrior, HistogramDensity
from gapmoe.parameterizations import (
    BinaryCircularParameterization,
    calc_vEarth,
)

rng = np.random.default_rng(42)
repo_root

## Load pre-generated histogram files

In [ ]:
demo_dir = repo_root / "example" / "pre_runner_outputs" / "demo"
if not demo_dir.exists():
    raise FileNotFoundError(f"{demo_dir} not found. Run pre_runner.ipynb first.")

density = HistogramDensity.from_paths(
    demo_dir / "mass.dat",
    demo_dir / "rho.dat",
    demo_dir / "murel.dat",
)

## Set up event-specific context

Both `rho`-based parameterizations need two pieces of event-specific
information supplied at sampling time:

- `thS`: angular radius of the source star in mas  
- `vEarth`: heliocentric Earth velocity at the event peak in AU/yr,
  returned as `(v_N, v_E)` by `calc_vEarth(t0_jd, ra_deg, dec_deg)`

These do **not** appear as free parameters — they are fixed observational
quantities for a given event.

In [ ]:
# Hypothetical event peak at HJD 2460000, toward (ra, dec) = (270, -30)
t0_jd = 2460000.0
ra_deg = 270.0
dec_deg = -30.0

v_N, v_E = calc_vEarth(t0_jd, ra_deg, dec_deg)
print(f"vEarth = ({v_N:.3f}, {v_E:.3f}) AU/yr")

# Source angular radius: typical bulge source at ~8 kpc, stellar radius ~1 Rsun
thS = 0.5  # mas

ctx = {"thS": thS, "vEarth": (v_N, v_E)}

## Direct use of the parameterization

Before setting up MCMC it is useful to verify that `to_physical` and
`log_abs_det_jacobian` return sensible values at a known point.

In [ ]:
param = BinaryCircularParameterization()
print("Parameter names:", param.names)

# A plausible binary-lens light-curve parameter vector
theta_trial = np.array([
    t0_jd,   # t0
    50.0,    # tE  [day]
    0.1,     # u0
    0.005,   # rho
    0.1,     # q
    1.0,     # s
    0.5,     # alpha  [rad]
    0.1,     # piEN
    0.05,    # piEE
    0.001,   # gamma1
    0.002,   # gamma2
    -0.003,  # gamma3
])

ML, DL, DS, mu_N, mu_E = param.to_physical(theta_trial, ctx)
print(f"ML   = {ML:.4f} Msun")
print(f"DL   = {DL:.4f} kpc")
print(f"DS   = {DS:.4f} kpc")
print(f"mu_N = {mu_N:.4f} mas/yr")
print(f"mu_E = {mu_E:.4f} mas/yr")

lndet = param.log_abs_det_jacobian(theta_trial, ctx)
print(f"log|det J| = {lndet:.6f}")

## Attach the parameterization to the prior

Pass `parameterization=param` to `GalacticPrior`.
The prior then accepts `(theta, ctx)` directly — the Jacobian correction is
applied automatically so the returned log-probability is in light-curve
parameter space.

In [ ]:
prior = GalacticPrior(density, parameterization=param)

lp_trial = prior.log_prob(theta_trial, context=ctx)
print(f"log prior (light-curve space) = {lp_trial:.4f}")

## MCMC in light-curve parameter space

Define bounds that reflect physically reasonable ranges for each light-curve
parameter. The sampler explores the 12-dimensional light-curve space while
the prior evaluates density in physical space via the parameterization.

> For a real event, add `log_likelihood(theta)` to `log_probability`.

In [ ]:
bounds = [
    (t0_jd - 200.0, t0_jd + 200.0),  # t0
    (1.0,   500.0),                   # tE  [day]
    (-2.0,   2.0),                    # u0
    (1e-4,  0.1),                     # rho
    (1e-4,  1.0),                     # q
    (0.1,   3.0),                     # s
    (0.0,   2 * np.pi),               # alpha
    (-1.0,  1.0),                     # piEN
    (-1.0,  1.0),                     # piEE
    (-0.1,  0.1),                     # gamma1
    (-0.1,  0.1),                     # gamma2
    (-0.1,  0.1),                     # gamma3
]


def in_bounds(theta):
    return all(lo <= v <= hi for v, (lo, hi) in zip(theta, bounds))


def log_probability(theta):
    if not in_bounds(theta):
        return -np.inf
    return prior.log_prob(theta, context=ctx)

In [ ]:
def draw_initial_positions(nwalkers, max_tries=200_000):
    positions = []
    while len(positions) < nwalkers and max_tries > 0:
        max_tries -= 1
        theta = np.array([rng.uniform(lo, hi) for lo, hi in bounds])
        if np.isfinite(log_probability(theta)):
            positions.append(theta)
    if len(positions) != nwalkers:
        raise RuntimeError("Could not find enough finite initial walker positions.")
    return np.array(positions)


ndim = len(bounds)
nwalkers = 32
initial = draw_initial_positions(nwalkers)
print(f"Found {nwalkers} finite starting positions.")

In [ ]:
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
sampler.run_mcmc(initial, 1000, progress=True)

print(f"Mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.3f}")

## Extract physical parameters from the chain

After thinning and burn-in, convert each sample in the chain to physical
parameters using `to_physical`.

In [ ]:
flat_chain = sampler.get_chain(discard=200, thin=5, flat=True)
print(f"Flat chain shape: {flat_chain.shape}")

phys_samples = np.array([
    param.to_physical(theta, ctx) for theta in flat_chain
])
phys_labels = ["ML", "DL", "DS", "mu_N", "mu_E"]

for i, label in enumerate(phys_labels):
    q16, q50, q84 = np.percentile(phys_samples[:, i], [16, 50, 84])
    print(f"{label:>4s}: {q50:.4g} -{q50 - q16:.4g} +{q84 - q50:.4g}")

In [ ]:
import corner

corner.corner(phys_samples, labels=phys_labels)
plt.show()